# 12F-11 — Canonical database ingestion

**Question:** How can the relational slice be loaded and checked safely?

The target is an explicit disposable database under `notebooks/_data/`, never a runtime DB.

In [ ]:
from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in search_roots if (path / "AGENTS.md").exists() and (path / "etl").exists())
import sqlite3

DATA_DIR = REPO_ROOT / "notebooks/_data"
SCHEMA_FILE = REPO_ROOT / "etl/schema/canonical_schema_v1.sql"
DEMO_DB = DATA_DIR / "12f11_canonical_notebook_demo.db"
input_paths = {name: DATA_DIR / f"12f10_{name}.csv" for name in ["program", "vehicle_configuration", "vde", "run", "fuelcons", "fuelcons_run_adoption"]}
assert SCHEMA_FILE.exists() and all(path.exists() for path in input_paths.values())


## Create only the guarded disposable target

In [ ]:
assert DEMO_DB.parent.resolve() == DATA_DIR.resolve()
assert DEMO_DB.name == "12f11_canonical_notebook_demo.db"
if DEMO_DB.exists():
    DEMO_DB.unlink()

connection = sqlite3.connect(DEMO_DB)
connection.executescript(SCHEMA_FILE.read_text(encoding="utf-8"))
connection.execute("PRAGMA foreign_keys=ON")


## Load in foreign-key-safe order

In [ ]:
load_order = ["program", "vehicle_configuration", "vde", "run", "fuelcons", "fuelcons_run_adoption"]
loaded_counts = {}
for table in load_order:
    frame = pd.read_csv(input_paths[table], low_memory=False)
    frame = frame.astype(object).where(pd.notna(frame), None)
    frame.to_sql(table, connection, if_exists="append", index=False)
    loaded_counts[table] = len(frame)
connection.commit()
adoption_count = connection.execute("SELECT COUNT(*) FROM fuelcons_run_adoption").fetchone()[0]
assert adoption_count > 0
assert adoption_count == loaded_counts["fuelcons_run_adoption"]
pd.Series(loaded_counts, name="loaded_rows")


## Integrity and a readable join

In [ ]:
foreign_key_violations = connection.execute("PRAGMA foreign_key_check").fetchall()
quick_check = connection.execute("PRAGMA quick_check").fetchone()[0]
assert foreign_key_violations == []
assert quick_check == "ok"
print("foreign_key_check: 0 violations")
print("quick_check:", quick_check)


In [ ]:
query = '''
SELECT p.commercial_make, p.commercial_model,
       vc.vehicle_configuration_id, v.id AS vde_id,
       COUNT(DISTINCT f.id) AS fuelcons_records,
       COUNT(DISTINCT a.run_id) AS adopted_runs
FROM program p
JOIN vehicle_configuration vc ON vc.program_id=p.program_id
JOIN vde v ON v.vehicle_configuration_id=vc.vehicle_configuration_id
LEFT JOIN fuelcons f ON f.vde_id=v.id
LEFT JOIN fuelcons_run_adoption a ON a.fuelcons_id=f.id AND a.vde_id=v.id
GROUP BY p.program_id, vc.vehicle_configuration_id, v.id
ORDER BY adopted_runs DESC
LIMIT 10
'''
joined = pd.read_sql_query(query, connection)
joined


In [ ]:
database_counts = {table: connection.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0] for table in load_order}
assert database_counts == loaded_counts
connection.close()
DEMO_DB.relative_to(REPO_ROOT)


## What did we learn?

Safe ingestion needs a disposable target, schema-first creation, dependency order, row-count reconciliation, and foreign-key/integrity checks. Runtime databases were never opened for writing.